In [1]:
import os

os.chdir("/orcd/archive/abugoot/001/Projects/paolo/tde_main/")

import sys
import argparse
import logging
import yaml
import json
import itertools
import copy
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import joblib

from utils.snapMMD import MMDLoss, RBF
from utils.experiment_utils import load_best_model, get_experiment_info
import hydra
from omegaconf import OmegaConf

from TrajectoryNet.optimal_transport.emd import earth_mover_distance

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [2]:
DATASET_CONFIGS: Dict[str, Dict[str, Any]] = {
    'LV': {
        'data_path': 'data/classic/LV_data.npz',
        'dimensionality': 2,
        'axes_labels': ['Prey', 'Predator'],
        'title': 'Lotka-Volterra',
        'calculate_emd': True,
    },
    'Repressilator': {
        'data_path': 'data/classic/Repressilator_data.npz',
        'dimensionality': 3,
        'axes_labels': ['Gene 1', 'Gene 2', 'Gene 3'],
        'title': 'Repressilator',
        'calculate_emd': True,
    },
    'GoM': {
        'data_path': 'data/realdata/GoM_data.npz',
        'dimensionality': 2,
        'axes_labels': ['X1', 'X2'],
        'title': 'GoM',
        'calculate_emd': True,
    },
    'pbmc': {
        'data_path': 'data/realdata/processed_pbmc_data_sub500_every_2_until20.npz',
        'dimensionality': 30,
        'plot_dimensionality': 3,
        'axes_labels': ['PC1', 'PC2', 'PC3'],
        'title': 'PBMC',
        'calculate_emd': False,
        'requires_pca': True,
    },
}

In [3]:
def load_cfg_and_ckpt(ckpt_dir):
    # Resolve and validate checkpoint directory
    experiment_dir = os.path.abspath(os.path.expanduser(str(ckpt_dir)))
    cfg_path = os.path.join(experiment_dir, 'config.yaml')
    ckpt_path = os.path.join(experiment_dir, 'best_model.pt')
    # Load trained config and use it as the active config
    cfg = OmegaConf.load(cfg_path)
    return cfg, ckpt_path

def load_models(cfg,ckpt_path):
    encoder = hydra.utils.instantiate(cfg.encoder).to(device)
    generator = hydra.utils.instantiate(cfg.generator).to(device)

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)

    encoder.load_state_dict(checkpoint['encoder_state_dict'])
    generator.load_state_dict(checkpoint['generator_state_dict'])
    encoder.eval()
    generator.eval()

    return encoder, generator


In [25]:
def generate_cde_forecast(cfg, data, encoder, generator):

    Xs = data['Xs']
    n_steps = int(data['N_steps'])
    
    print(Xs.shape,n_steps)

    set_size = cfg.experiment.set_size

    Xs_second_last = torch.tensor(Xs[-2], dtype=torch.float).to(device)
    Xs_last = torch.tensor(Xs[-1], dtype=torch.float).to(device)
    
    num_sets = Xs_second_last.shape[0] // set_size
    
    all_gen_samples = []
    
    
    for i in range(num_sets):
        Xs_second_last_set = Xs_second_last[i*set_size:(i+1)*set_size].unsqueeze(0)
        Xs_last_set = Xs_last[i*set_size:(i+1)*set_size].unsqueeze(0)
        
        #print("inputs",Xs_second_last_set.shape, Xs_last_set.shape)

        src_latent = encoder(Xs_second_last_set)
        tgt_latent = encoder(Xs_last_set)
        
        #print(src_latent.shape, tgt_latent.shape)

        gen = generator.sample(Xs_second_last_set.squeeze(0), src_latent, tgt_latent)

        #print("outputs",gen.shape)
        
        all_gen_samples.append(gen.squeeze(0))
    
    all_gen_samples = torch.cat(all_gen_samples, dim=0)
    
    return all_gen_samples





    

In [26]:
# NOTE: THIS IS OUR METHOD (TDEs)

ckpt_dir = "./outputs/snapMMD_energy_cotrain_LV_ed19dd974fb0e42004bbacc8b55bf4c1"

cfg, ckpt_path = load_cfg_and_ckpt(ckpt_dir)
encoder, generator = load_models(cfg, ckpt_path)

data = np.load(DATASET_CONFIGS[cfg.dataset_name]['data_path'])

forecast = generate_cde_forecast(cfg, data, encoder, generator)
print(forecast.shape)

(11, 200, 2) 11
torch.Size([192, 2])
